# NDWI Difference Calculation (Report Files)

In [ ]:
# ----------------------------
# ndwi_change_analysis_multi_sensor.py
# ----------------------------
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import os
import glob
import pandas as pd

In [ ]:
# ----------------------------
# USER INPUT: file paths for all sensors and years
# ----------------------------
# FILES = {
#     "landsat8": {
#         "ndwi": {
#             "2016": "../data/results/landsat8_ndwi_2016.tif",
#             "2021": "../data/results/landsat8_ndwi_2021.tif"
#         },
#         "mask": {
#             "2016": "../data/results/landsat8_watermask_2016.tif",
#             "2021": "../data/results/landsat8_watermask_2021.tif"
#         }
#     },
#     "sentinel2": {
#         "ndwi": {
#             "2016": "../data/results/sentinel2_ndwi_2016.tif",
#             "2021": "../data/results/sentinel2_ndwi_2021.tif"
#         },
#         "mask": {
#             "2016": "../data/results/sentinel2_watermask_2016.tif",
#             "2021": "../data/results/sentinel2_watermask_2021.tif"
#         }
#     }
# }


INPUT_DIR = "../data/results"
sensors = ["landsat8", "sentinel2"]
expected_years = ["2016", "2017", "2018", "2019", "2020", "2021"]

FILES = {}

for sensor in sensors:
    sensor_folder = os.path.join(INPUT_DIR, sensor)
    FILES[sensor] = {"ndwi": {}, "mask": {}}

    # NDWI files
    ndwi_files = glob.glob(os.path.join(sensor_folder, f"{sensor}_ndwi_*.tif"))
    for f in ndwi_files:
        year = os.path.basename(f).split("_")[-1].split(".")[0]
        FILES[sensor]["ndwi"][year] = f

    # Water mask files
    mask_files = glob.glob(os.path.join(sensor_folder, f"{sensor}_watermask_*.tif"))
    for f in mask_files:
        year = os.path.basename(f).split("_")[-1].split(".")[0]
        FILES[sensor]["mask"][year] = f

    # Check for missing expected years
    for year in expected_years:
        if year not in FILES[sensor]["ndwi"]:
            print(f"WARNING: NDWI file missing for {sensor}, year {year}")
        if year not in FILES[sensor]["mask"]:
            print(f"WARNING: Water mask file missing for {sensor}, year {year}")

# Optional: pretty print the resulting dictionary
import pprint
pprint.pprint(FILES)

OUTPUT_DIR = "../data/results/report"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [3]:
# ----------------------------
# Helpers
# ----------------------------
def reproject_to_target(src_path, target_crs, target_transform=None, target_width=None, target_height=None):
    with rasterio.open(src_path) as src:
        src_profile = src.profile
        if target_transform is None:
            transform, width, height = calculate_default_transform(
                src.crs, target_crs, src.width, src.height, *src.bounds
            )
        else:
            transform, width, height = target_transform, target_width, target_height

        dst_profile = src_profile.copy()
        dst_profile.update({
            "crs": target_crs,
            "transform": transform,
            "width": width,
            "height": height
        })

        bands = src.count
        dst = np.zeros((bands, height, width), dtype=src.dtypes[0])
        for i in range(1, bands + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=dst[i-1],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=target_crs,
                resampling=Resampling.bilinear if bands > 1 else Resampling.nearest
            )
    if dst.shape[0] == 1:
        arr = dst[0]
    else:
        arr = np.transpose(dst, (1, 2, 0))
    return arr, transform, target_crs, dst_profile

def reproject_mask_to_grid(src_path, target_crs, target_transform, target_width, target_height):
    with rasterio.open(src_path) as src:
        dst = np.zeros((1, target_height, target_width), dtype=src.dtypes[0])
        reproject(
            source=rasterio.band(src, 1),
            destination=dst[0],
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=target_transform,
            dst_crs=target_crs,
            resampling=Resampling.nearest
        )
    return dst[0]

def compute_change(ndwi_2016, ndwi_2021, mask_2016, mask_2021):
    ndwi_diff = ndwi_2021 - ndwi_2016
    nodata_mask = np.isnan(ndwi_2016) | np.isnan(ndwi_2021)
    ndwi_diff_masked = np.where(nodata_mask, np.nan, ndwi_diff)

    wm16 = np.where(mask_2016 > 0.5, 1, 0)
    wm21 = np.where(mask_2021 > 0.5, 1, 0)
    change_map = (wm21 - wm16).astype(int)
    change_map[nodata_mask] = 0
    
# ----------------------------
# compute summary stats for water change
# ----------------------------
# Pixel area from transform (meters, EPSG:3857)
    res_x = abs(tform.a)  # pixel width
    res_y = abs(tform.e)  # pixel height
    pixel_area = res_x * res_y  # m^2 per pixel

    # Count gain/loss pixels
    num_gain_pixels = np.sum(change_map == 1)
    num_loss_pixels = np.sum(change_map == -1)

    # Compute area in km^2
    area_gain_km2 = num_gain_pixels * pixel_area / 1e6
    area_loss_km2 = num_loss_pixels * pixel_area / 1e6
    net_change_km2 = area_gain_km2 - area_loss_km2

    # Print summary
    print("Summary of Water Change:")
    print(f"Area gained (km²): {area_gain_km2:.3f}")
    print(f"Area lost   (km²): {area_loss_km2:.3f}")
    print(f"Net change  (km²): {net_change_km2:.3f}")
    print(f"Gain pixels: {num_gain_pixels}")
    print(f"Loss pixels: {num_loss_pixels}")

    
    return ndwi_diff_masked, change_map

# ----------------------------
# Main loop: process each sensor
# ----------------------------
for sensor, data in FILES.items():
    print(f"Processing {sensor}...")

    # Get CRS from 2021 NDWI
    with rasterio.open(data["ndwi"]["2021"]) as ref_src:
        TARGET_CRS = ref_src.crs

    # Reproject 2021 NDWI
    ndwi_2021_arr, ref_transform, ref_crs, ref_profile = reproject_to_target(data["ndwi"]["2021"], TARGET_CRS)

    # Get target grid dimensions
    with rasterio.open(data["ndwi"]["2021"]) as tmp_src:
        tform, w, h = calculate_default_transform(tmp_src.crs, TARGET_CRS, tmp_src.width, tmp_src.height, *tmp_src.bounds)

    # Reproject 2016 NDWI
    ndwi_2016_arr, _, _, _ = reproject_to_target(data["ndwi"]["2016"], TARGET_CRS, target_transform=tform, target_width=w, target_height=h)

    # Reproject masks
    mask_2021_arr = reproject_mask_to_grid(data["mask"]["2021"], TARGET_CRS, tform, w, h)
    mask_2016_arr = reproject_mask_to_grid(data["mask"]["2016"], TARGET_CRS, tform, w, h)

    # Convert NDWI arrays to float
    ndwi_2016_arr = ndwi_2016_arr.astype(float)
    ndwi_2021_arr = ndwi_2021_arr.astype(float)

    # Compute change
    ndwi_diff, change_map = compute_change(ndwi_2016_arr, ndwi_2021_arr, mask_2016_arr, mask_2021_arr)

    # ----------------------------
    # Plot results for slides
    # ----------------------------
    fig, axes = plt.subplots(1,3, figsize=(18,6))

    # NDWI diff
    im = axes[0].imshow(ndwi_diff, cmap="bwr", vmin=-0.5, vmax=0.5)
    axes[0].set_title(f"{sensor} NDWI difference (2021-2016)")
    plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

    # Water change map
    cmap_change = ListedColormap(["red","lightgray","blue"])
    viz_change = change_map + 1  # map -1,0,1 -> 0,1,2
    axes[1].imshow(viz_change, cmap=cmap_change, vmin=0, vmax=2)
    axes[1].set_title("Water change: red=loss, blue=gain")
    axes[1].axis("off")

    # NDWI 2021 with contours
    axes[2].imshow(ndwi_2021_arr, cmap="Blues")
    axes[2].set_title(f"{sensor} 2021 NDWI with change contours")
    import scipy.ndimage as ndi
    axes[2].contour(change_map==1, levels=[0.5], colors="cyan", linewidths=0.8)
    axes[2].contour(change_map==-1, levels=[0.5], colors="magenta", linewidths=0.8)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{sensor}_ndwi_change.png"), dpi=300)
    plt.close(fig)
    print(f"Saved plots for {sensor} to {OUTPUT_DIR}")


Processing landsat8...
Summary of Water Change:
Area gained (km²): 0.001
Area lost   (km²): 4.360
Net change  (km²): -4.359
Gain pixels: 1
Loss pixels: 4844
Saved plots for landsat8 to ../data/results/slides
Processing sentinel2...
Summary of Water Change:
Area gained (km²): 0.160
Area lost   (km²): 0.914
Net change  (km²): -0.754
Gain pixels: 1602
Loss pixels: 9140
Saved plots for sentinel2 to ../data/results/slides


In [ ]:
# -----------------------------
# Config
# -----------------------------
OUTPUT_DIR = "../data/results"
sensors = ["landsat8", "sentinel2"]
expected_years = ["2016", "2017", "2018", "2019", "2020", "2021"]
ndwi_threshold = 0.3         # threshold for binary water mask
delta_threshold = 0.05       # minimal NDWI change to consider gain/loss

FILES = {}
summary_rows = []

# -----------------------------
# Build file dictionary & check missing files
# -----------------------------
for sensor in sensors:
    sensor_folder = os.path.join(OUTPUT_DIR, sensor)
    FILES[sensor] = {"ndwi": {}, "mask": {}}

    for f in glob.glob(os.path.join(sensor_folder, f"{sensor}_ndwi_*.tif")):
        year = os.path.basename(f).split("_")[-1].split(".")[0]
        FILES[sensor]["ndwi"][year] = f

    for f in glob.glob(os.path.join(sensor_folder, f"{sensor}_watermask_*.tif")):
        year = os.path.basename(f).split("_")[-1].split(".")[0]
        FILES[sensor]["mask"][year] = f

    # Warn if files missing
    for year in expected_years:
        if year not in FILES[sensor]["ndwi"]:
            print(f"WARNING: NDWI missing for {sensor}, year {year}")
        if year not in FILES[sensor]["mask"]:
            print(f"WARNING: Water mask missing for {sensor}, year {year}")

# -----------------------------
# Temporal Change Analysis + Statistics
# -----------------------------
for sensor in sensors:
    print(f"\nProcessing sensor: {sensor}")
    sensor_folder = os.path.join(OUTPUT_DIR, sensor)
    os.makedirs(sensor_folder, exist_ok=True)

    # Load NDWI rasters
    ndwi_stack = []
    years_available = sorted(FILES[sensor]["ndwi"].keys())
    for year in years_available:
        with rasterio.open(FILES[sensor]["ndwi"][year]) as src:
            ndwi_stack.append(src.read(1))
    ndwi_stack = np.array(ndwi_stack)  # shape: (years, rows, cols)

    # Compute binary water masks for persistence
    binary_stack = ndwi_stack > ndwi_threshold
    persistence = np.sum(binary_stack, axis=0)

    # Save multi-year persistence map
    plt.figure(figsize=(8,6))
    plt.imshow(persistence, cmap='viridis')
    plt.colorbar(label="Years with Water")
    plt.title(f"{sensor} Multi-Year Water Persistence")
    out_file = os.path.join(sensor_folder, f"{sensor}_water_persistence.png")
    plt.savefig(out_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved persistence map: {out_file}")

    # Year-to-Year ΔNDWI Maps + stats
    for i in range(1, len(years_available)):
        year_prev = years_available[i-1]
        year_curr = years_available[i]
        delta = ndwi_stack[i] - ndwi_stack[i-1]

        # Gain / Loss masks
        gain = delta > delta_threshold
        loss = delta < -delta_threshold

        # Compute statistics
        pixel_area_km2 = 0.0009 if sensor=="landsat8" else 0.0001  # 30m^2 vs 10m^2
        area_gain = np.sum(gain) * pixel_area_km2
        area_loss = np.sum(loss) * pixel_area_km2
        net_change = area_gain - area_loss
        summary_rows.append({
            "Sensor": sensor,
            "Year_Prev": year_prev,
            "Year_Curr": year_curr,
            "Area_Gained_km2": area_gain,
            "Area_Lost_km2": area_loss,
            "Net_Change_km2": net_change,
            "Gain_Pixels": np.sum(gain),
            "Loss_Pixels": np.sum(loss)
        })

        # Plot change map with water mask overlay
        plt.figure(figsize=(8,6))
        plt.imshow(delta, cmap='bwr', vmin=-0.5, vmax=0.5)
        plt.contour(binary_stack[i], colors='blue', linewidths=0.5, alpha=0.5)
        plt.contour(binary_stack[i-1], colors='red', linewidths=0.5, alpha=0.3)
        plt.title(f"{sensor} ΔNDWI: {year_prev} → {year_curr}")
        plt.colorbar(label="ΔNDWI")
        out_file = os.path.join(sensor_folder, f"{sensor}_ndwi_change_{year_prev}_{year_curr}.png")
        plt.savefig(out_file, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Saved change map: {out_file}")

# -----------------------------
# Save summary CSV
# -----------------------------
summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(OUTPUT_DIR, "surface_water_change_summary.csv")
summary_df.to_csv(summary_csv, index=False)
print(f"\nSaved quantitative summary: {summary_csv}")
